# SVM–TMC fusion across five folds and seeds
This is a clean JHub-native follow-up notebook for the final matched comparison.

It uses:

- outer folds: **0, 1, 2, 3, 4**
- matched seeds: **17, 42, 73, 101, 137**
- RBF-SVM predictions from the completed five-seed SVM run
- TMC predictions from the medium neural architecture
- no retraining

## What this notebook reports

There are two complementary summaries.

### 1. Five independent matched-seed runs
For each seed separately:

1. concatenate the five held-out outer-test folds to obtain 544 OOF participants;
2. calculate SVM, TMC and 50/50 SVM+TMC ROC AUC;
3. summarize mean ± SD across the five seeds.

This is the fairest seed-sensitivity comparison.

### 2. Five-seed probability ensemble
Within each fold, average SVM probabilities across the five seeds and average TMC probabilities across the five seeds. Then:

- evaluate SVM ensemble;
- evaluate TMC ensemble;
- evaluate **50/50 SVM + TMC ensemble**;
- concatenate the five test folds into one 544-participant OOF set.

This is the direct five-seed version of the earlier three-seed fusion result.

### Validation-selected alpha
The notebook also repeats the leakage-aware continuous-weight experiment using all five seeds:

\[
p_{fusion}=\alpha p_{SVM}+(1-\alpha)p_{TMC}
\]

For each outer fold, `alpha` is selected only on that fold's **validation set** after averaging the five seeds, then locked and applied to the test fold.

The previous three-seed notebook is left unchanged for reproducibility.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# Experiment contract

FOLDS = [0, 1, 2, 3, 4]
SEEDS = [17, 42, 73, 101, 137]
THRESHOLD = 0.50

ALPHA_GRID = np.round(
    np.arange(0.0, 1.0001, 0.05),
    2,
)

N_BOOTSTRAP = 3000
BOOTSTRAP_SEED = 20261005

# JHub paths

RUNS_ROOT = Path("/home/jovyan/thesis_runs")

SVM_PREDICTIONS_DIR = (
    RUNS_ROOT
    / "svm_rbf_predictions_5seeds"
    / "fold_predictions"
)

TMC_ROOT = (
    RUNS_ROOT
    / "gated_cmt_fixed_equal_fusion_medium_v1_md050"
)

OUTPUT_ROOT = (
    RUNS_ROOT
    / "svm_tmc_5seeds_5folds_fusion_v1"
)

TABLES_DIR = OUTPUT_ROOT / "tables"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"
FIGURES_DIR = OUTPUT_ROOT / "figures"

for d in [
    OUTPUT_ROOT,
    TABLES_DIR,
    PREDICTIONS_DIR,
    FIGURES_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("SVM predictions:", SVM_PREDICTIONS_DIR)
print("TMC root:", TMC_ROOT)
print("Output:", OUTPUT_ROOT)
print("Folds:", FOLDS)
print("Seeds:", SEEDS)
print("Alpha grid:", ALPHA_GRID.tolist())


## 1. Input check

Required inputs:

- 25 SVM test files;
- 25 SVM validation files;
- 25 TMC test files;
- 25 TMC validation files.


In [ ]:
missing = []

for fold in FOLDS:
    for seed in SEEDS:
        svm_test = (
            SVM_PREDICTIONS_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
        )

        svm_val = (
            SVM_PREDICTIONS_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_validation_predictions.csv"
        )

        tmc_test = (
            TMC_ROOT
            / f"seed_{seed}"
            / "mci_prognosis"
            / f"fold_{fold}"
            / "predictions"
            / "test_predictions.csv"
        )

        tmc_val = (
            TMC_ROOT
            / f"seed_{seed}"
            / "mci_prognosis"
            / f"fold_{fold}"
            / "predictions"
            / "best_validation_predictions.csv"
        )

        for path in [
            svm_test,
            svm_val,
            tmc_test,
            tmc_val,
        ]:
            if not path.exists():
                missing.append(path)

if missing:
    print("Missing required files:")
    for p in missing:
        print(" -", p)

    raise FileNotFoundError(
        f"{len(missing)} required file(s) are missing."
    )

print("Input check passed.")
print("All 100 required fold/seed prediction files are present.")


## 2. Load one matched fold/seed pair

The loader verifies participant IDs and targets before any fusion is computed.


In [ ]:
def require_columns(df, cols, context):
    missing = [
        c for c in cols
        if c not in df.columns
    ]

    if missing:
        raise KeyError(
            f"{context}: missing columns {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )

def svm_path(
    fold,
    seed,
    split,
):
    suffix = (
        "test_predictions.csv"
        if split == "test"
        else "validation_predictions.csv"
    )

    return (
        SVM_PREDICTIONS_DIR
        / f"svm_rbf_fold_{fold}_seed_{seed}_{suffix}"
    )

def tmc_path(
    fold,
    seed,
    split,
):
    filename = (
        "test_predictions.csv"
        if split == "test"
        else "best_validation_predictions.csv"
    )

    return (
        TMC_ROOT
        / f"seed_{seed}"
        / "mci_prognosis"
        / f"fold_{fold}"
        / "predictions"
        / filename
    )

def load_matched(
    fold,
    seed,
    split,
):
    s_path = svm_path(
        fold,
        seed,
        split,
    )

    t_path = tmc_path(
        fold,
        seed,
        split,
    )

    svm = pd.read_csv(
        s_path
    )

    tmc = pd.read_csv(
        t_path
    )

    require_columns(
        svm,
        [
            "RID",
            "TARGET",
            "PROB_PMCI",
        ],
        str(s_path),
    )

    require_columns(
        tmc,
        [
            "RID",
            "TARGET",
            "TMC_P_pMCI",
        ],
        str(t_path),
    )

    svm = svm[
        [
            "RID",
            "TARGET",
            "PROB_PMCI",
        ]
    ].rename(
        columns={
            "PROB_PMCI":
                "SVM_P"
        }
    )

    tmc_cols = [
        "RID",
        "TARGET",
        "TMC_P_pMCI",
    ]

    if "TMC_UNCERTAINTY" in tmc.columns:
        tmc_cols.append(
            "TMC_UNCERTAINTY"
        )

    tmc = tmc[
        tmc_cols
    ].rename(
        columns={
            "TMC_P_pMCI":
                "TMC_P"
        }
    )

    merged = tmc.merge(
        svm,
        on="RID",
        how="inner",
        suffixes=(
            "_TMC",
            "_SVM",
        ),
        validate="one_to_one",
    )

    if len(merged) != len(tmc):
        raise ValueError(
            f"fold={fold} seed={seed} split={split}: "
            "SVM and TMC participant sets do not match."
        )

    if not (
        merged["TARGET_TMC"].astype(int)
        == merged["TARGET_SVM"].astype(int)
    ).all():
        raise ValueError(
            f"fold={fold} seed={seed} split={split}: "
            "target mismatch between SVM and TMC."
        )

    merged = (
        merged
        .rename(
            columns={
                "TARGET_TMC":
                    "TARGET"
            }
        )
        .drop(
            columns=[
                "TARGET_SVM"
            ]
        )
    )

    merged["FOLD"] = fold
    merged["SEED"] = seed
    merged["SPLIT"] = split

    return merged

example = load_matched(
    fold=0,
    seed=17,
    split="test",
)

display(
    example.head()
)

print("Rows:", len(example))


## 3. Metrics


In [ ]:
def metrics_from_prob(
    y_true,
    y_prob,
    threshold=THRESHOLD,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float,
    )

    pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn)
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp)
        else np.nan
    )

    return {
        "ROC_AUC":
            float(
                roc_auc_score(
                    y_true,
                    y_prob,
                )
            ),

        "AP":
            float(
                average_precision_score(
                    y_true,
                    y_prob,
                )
            ),

        "BALANCED_ACCURACY":
            float(
                balanced_accuracy_score(
                    y_true,
                    pred,
                )
            ),

        "SENSITIVITY":
            float(sensitivity),

        "SPECIFICITY":
            float(specificity),

        "F1":
            float(
                f1_score(
                    y_true,
                    pred,
                )
            ),

        "BRIER":
            float(
                brier_score_loss(
                    y_true,
                    y_prob,
                )
            ),

        "N":
            int(len(y_true)),
    }


## 4. Matched seed-by-seed 5-fold OOF analysis

For each seed independently, the five held-out test folds are concatenated to 544 participants.

This gives five comparable OOF AUCs for:

- SVM
- TMC
- fixed 50/50 SVM+TMC


In [ ]:
seed_oof_frames = []
seed_metric_rows = []

for seed in SEEDS:
    fold_parts = []

    for fold in FOLDS:
        g = load_matched(
            fold=fold,
            seed=seed,
            split="test",
        )

        g[
            "P_EQUAL_SVM_TMC"
        ] = (
            0.5 * g["SVM_P"]
            + 0.5 * g["TMC_P"]
        )

        fold_parts.append(
            g
        )

    pooled = pd.concat(
        fold_parts,
        ignore_index=True,
    )

    if (
        len(pooled) != 544
        or pooled["RID"].nunique() != 544
    ):
        raise ValueError(
            f"Seed {seed}: expected 544 unique pooled OOF participants."
        )

    seed_oof_frames.append(
        pooled
    )

    for method, col in {
        "svm":
            "SVM_P",

        "tmc":
            "TMC_P",

        "equal_svm_tmc":
            "P_EQUAL_SVM_TMC",
    }.items():
        m = metrics_from_prob(
            pooled["TARGET"],
            pooled[col],
        )

        seed_metric_rows.append({
            "SEED":
                seed,

            "METHOD":
                method,

            **m,
        })

        pooled[
            [
                "RID",
                "TARGET",
                "FOLD",
                "SEED",
                "SVM_P",
                "TMC_P",
                "P_EQUAL_SVM_TMC",
            ]
        ].to_csv(
            PREDICTIONS_DIR
            / f"seed_{seed}_matched_5fold_oof_predictions.csv",
            index=False,
        )

seed_metrics = pd.DataFrame(
    seed_metric_rows
)

display(
    seed_metrics.pivot(
        index="SEED",
        columns="METHOD",
        values="ROC_AUC",
    )
)

seed_metrics.to_csv(
    TABLES_DIR
    / "matched_seed_pooled_oof_metrics.csv",
    index=False,
)


## 5. Mean ± SD across the five matched seeds

This is the seed-sensitivity summary.


In [ ]:
seed_summary_rows = []

for method in [
    "svm",
    "tmc",
    "equal_svm_tmc",
]:
    g = seed_metrics.loc[
        seed_metrics["METHOD"] == method
    ]

    seed_summary_rows.append({
        "METHOD":
            method,

        "N_SEEDS":
            len(g),

        "ROC_AUC_MEAN":
            g["ROC_AUC"].mean(),

        "ROC_AUC_SD":
            g["ROC_AUC"].std(
                ddof=1
            ),

        "AP_MEAN":
            g["AP"].mean(),

        "AP_SD":
            g["AP"].std(
                ddof=1
            ),

        "BALANCED_ACCURACY_MEAN":
            g["BALANCED_ACCURACY"].mean(),

        "BALANCED_ACCURACY_SD":
            g["BALANCED_ACCURACY"].std(
                ddof=1
            ),

        "BRIER_MEAN":
            g["BRIER"].mean(),

        "BRIER_SD":
            g["BRIER"].std(
                ddof=1
            ),
    })

seed_summary = pd.DataFrame(
    seed_summary_rows
).sort_values(
    "ROC_AUC_MEAN",
    ascending=False,
)

display(
    seed_summary
)

seed_summary.to_csv(
    TABLES_DIR
    / "matched_5seed_mean_sd.csv",
    index=False,
)


## 6. Build five-seed ensemble validation and test probabilities

Within each fold and split, the five SVM probabilities are averaged and the five TMC probabilities are averaged before fusion.


In [ ]:
def build_5seed_ensemble(
    fold,
    split,
):
    frames = []

    for seed in SEEDS:
        g = load_matched(
            fold=fold,
            seed=seed,
            split=split,
        )

        frames.append(
            g
        )

    raw = pd.concat(
        frames,
        ignore_index=True,
    )

    seed_counts = (
        raw
        .groupby("RID")["SEED"]
        .nunique()
    )

    if not (
        seed_counts == len(SEEDS)
    ).all():
        raise ValueError(
            f"fold={fold} split={split}: "
            "not every participant has all five seeds."
        )

    target_counts = (
        raw
        .groupby("RID")["TARGET"]
        .nunique()
    )

    if not (
        target_counts == 1
    ).all():
        raise ValueError(
            f"fold={fold} split={split}: "
            "target mismatch across seeds."
        )

    agg_dict = {
        "TARGET":
            ("TARGET", "first"),

        "SVM_P":
            ("SVM_P", "mean"),

        "SVM_P_SD":
            ("SVM_P", "std"),

        "TMC_P":
            ("TMC_P", "mean"),

        "TMC_P_SD":
            ("TMC_P", "std"),
    }

    if "TMC_UNCERTAINTY" in raw.columns:
        agg_dict[
            "TMC_UNCERTAINTY"
        ] = (
            "TMC_UNCERTAINTY",
            "mean",
        )

    out = (
        raw
        .groupby("RID")
        .agg(
            **agg_dict
        )
        .reset_index()
    )

    out["FOLD"] = fold
    out["SPLIT"] = split

    return out

validation_ensembles = {
    fold:
        build_5seed_ensemble(
            fold,
            "validation",
        )
    for fold in FOLDS
}

test_ensembles = {
    fold:
        build_5seed_ensemble(
            fold,
            "test",
        )
    for fold in FOLDS
}

print("Validation sizes:")
for fold in FOLDS:
    print(
        fold,
        len(validation_ensembles[fold]),
    )

print("\nTest sizes:")
for fold in FOLDS:
    print(
        fold,
        len(test_ensembles[fold]),
    )


## 7. Primary five-seed ensemble result: fixed 50/50 SVM + TMC

This is the direct replacement for the earlier three-seed `0.94387` result.


In [ ]:
ensemble_test_parts = []
ensemble_fold_rows = []

for fold in FOLDS:
    g = test_ensembles[
        fold
    ].copy()

    g[
        "P_EQUAL_SVM_TMC"
    ] = (
        0.5 * g["SVM_P"]
        + 0.5 * g["TMC_P"]
    )

    ensemble_test_parts.append(
        g
    )

    for method, col in {
        "svm_5seed_ensemble":
            "SVM_P",

        "tmc_5seed_ensemble":
            "TMC_P",

        "equal_svm_tmc_5seed_ensemble":
            "P_EQUAL_SVM_TMC",
    }.items():
        m = metrics_from_prob(
            g["TARGET"],
            g[col],
        )

        ensemble_fold_rows.append({
            "FOLD":
                fold,

            "METHOD":
                method,

            **m,
        })

ensemble_oof = pd.concat(
    ensemble_test_parts,
    ignore_index=True,
)

if (
    len(ensemble_oof) != 544
    or ensemble_oof["RID"].nunique() != 544
):
    raise ValueError(
        "Expected exactly 544 unique five-seed ensemble OOF participants."
    )

ensemble_pooled_rows = []

for method, col in {
    "svm_5seed_ensemble":
        "SVM_P",

    "tmc_5seed_ensemble":
        "TMC_P",

    "equal_svm_tmc_5seed_ensemble":
        "P_EQUAL_SVM_TMC",
}.items():
    m = metrics_from_prob(
        ensemble_oof["TARGET"],
        ensemble_oof[col],
    )

    ensemble_pooled_rows.append({
        "METHOD":
            method,

        **m,
    })

ensemble_pooled = (
    pd.DataFrame(
        ensemble_pooled_rows
    )
    .sort_values(
        "ROC_AUC",
        ascending=False,
    )
    .reset_index(drop=True)
)

ensemble_fold_metrics = pd.DataFrame(
    ensemble_fold_rows
)

print("PRIMARY FIVE-SEED ENSEMBLE RESULTS")
display(
    ensemble_pooled
)

print("\nFold ROC AUCs:")
display(
    ensemble_fold_metrics.pivot(
        index="FOLD",
        columns="METHOD",
        values="ROC_AUC",
    )
)

ensemble_oof.to_csv(
    PREDICTIONS_DIR
    / "five_seed_ensemble_5fold_oof_predictions.csv",
    index=False,
)

ensemble_pooled.to_csv(
    TABLES_DIR
    / "five_seed_ensemble_pooled_metrics.csv",
    index=False,
)

ensemble_fold_metrics.to_csv(
    TABLES_DIR
    / "five_seed_ensemble_fold_metrics.csv",
    index=False,
)


## 8. Validation-selected continuous SVM weight using all five seeds

For each outer fold:

- average all five SVM validation probabilities;
- average all five TMC validation probabilities;
- scan the pre-specified `alpha = 0.00, 0.05, ..., 1.00`;
- select by validation ROC AUC, then AP;
- lock the selected alpha;
- apply it once to the corresponding five-seed-ensemble test fold.

`alpha = 1` means pure SVM.  
`alpha = 0` means pure TMC.


In [ ]:
def select_alpha(
    validation_df,
):
    y = validation_df[
        "TARGET"
    ].to_numpy(
        dtype=int
    )

    svm_p = validation_df[
        "SVM_P"
    ].to_numpy(
        dtype=float
    )

    tmc_p = validation_df[
        "TMC_P"
    ].to_numpy(
        dtype=float
    )

    rows = []

    for alpha in ALPHA_GRID:
        p = (
            alpha * svm_p
            + (1.0 - alpha) * tmc_p
        )

        rows.append({
            "ALPHA":
                float(alpha),

            "VALIDATION_ROC_AUC":
                float(
                    roc_auc_score(
                        y,
                        p,
                    )
                ),

            "VALIDATION_AP":
                float(
                    average_precision_score(
                        y,
                        p,
                    )
                ),

            "DISTANCE_FROM_0P5":
                abs(
                    float(alpha)
                    - 0.5
                ),
        })

    ranking = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "VALIDATION_ROC_AUC",
                "VALIDATION_AP",
                "DISTANCE_FROM_0P5",
                "ALPHA",
            ],
            ascending=[
                False,
                False,
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )

    return (
        float(
            ranking.iloc[0][
                "ALPHA"
            ]
        ),
        ranking,
    )

alpha_rows = []

for fold in FOLDS:
    val = validation_ensembles[
        fold
    ]

    alpha, ranking = select_alpha(
        val
    )

    alpha_rows.append({
        "FOLD":
            fold,

        "SELECTED_ALPHA_SVM":
            alpha,

        "VALIDATION_SELECTED_FUSION_AUC":
            float(
                ranking.iloc[0][
                    "VALIDATION_ROC_AUC"
                ]
            ),

        "VALIDATION_SVM_AUC":
            float(
                roc_auc_score(
                    val["TARGET"],
                    val["SVM_P"],
                )
            ),

        "VALIDATION_TMC_AUC":
            float(
                roc_auc_score(
                    val["TARGET"],
                    val["TMC_P"],
                )
            ),
    })

    ranking.to_csv(
        TABLES_DIR
        / f"fold_{fold}_validation_alpha_ranking.csv",
        index=False,
    )

alpha_summary = pd.DataFrame(
    alpha_rows
)

display(
    alpha_summary
)

alpha_summary.to_csv(
    TABLES_DIR
    / "validation_selected_alpha_by_fold.csv",
    index=False,
)


## 9. Apply locked validation-selected alpha to test


In [ ]:
selected_test_parts = []
selected_fold_rows = []

for fold in FOLDS:
    g = test_ensembles[
        fold
    ].copy()

    alpha = float(
        alpha_summary.loc[
            alpha_summary["FOLD"]
            == fold,
            "SELECTED_ALPHA_SVM",
        ].iloc[0]
    )

    g[
        "P_VALIDATION_SELECTED_SVM_TMC"
    ] = (
        alpha * g["SVM_P"]
        + (1.0 - alpha)
        * g["TMC_P"]
    )

    g[
        "P_EQUAL_SVM_TMC"
    ] = (
        0.5 * g["SVM_P"]
        + 0.5 * g["TMC_P"]
    )

    g[
        "SELECTED_ALPHA_SVM"
    ] = alpha

    selected_test_parts.append(
        g
    )

    for method, col in {
        "svm_5seed_ensemble":
            "SVM_P",

        "tmc_5seed_ensemble":
            "TMC_P",

        "equal_svm_tmc_5seed_ensemble":
            "P_EQUAL_SVM_TMC",

        "validation_selected_svm_tmc_5seed_ensemble":
            "P_VALIDATION_SELECTED_SVM_TMC",
    }.items():
        m = metrics_from_prob(
            g["TARGET"],
            g[col],
        )

        selected_fold_rows.append({
            "FOLD":
                fold,

            "METHOD":
                method,

            **m,
        })

selected_oof = pd.concat(
    selected_test_parts,
    ignore_index=True,
)

selected_pooled_rows = []

selected_method_cols = {
    "svm_5seed_ensemble":
        "SVM_P",

    "tmc_5seed_ensemble":
        "TMC_P",

    "equal_svm_tmc_5seed_ensemble":
        "P_EQUAL_SVM_TMC",

    "validation_selected_svm_tmc_5seed_ensemble":
        "P_VALIDATION_SELECTED_SVM_TMC",
}

for method, col in selected_method_cols.items():
    m = metrics_from_prob(
        selected_oof["TARGET"],
        selected_oof[col],
    )

    selected_pooled_rows.append({
        "METHOD":
            method,

        **m,
    })

selected_pooled = (
    pd.DataFrame(
        selected_pooled_rows
    )
    .sort_values(
        "ROC_AUC",
        ascending=False,
    )
    .reset_index(drop=True)
)

selected_fold_metrics = pd.DataFrame(
    selected_fold_rows
)

print("FIVE-SEED ENSEMBLE + VALIDATION-SELECTED ALPHA")
display(
    selected_pooled
)

print("\nFold ROC AUCs:")
display(
    selected_fold_metrics.pivot(
        index="FOLD",
        columns="METHOD",
        values="ROC_AUC",
    )
)

selected_oof.to_csv(
    PREDICTIONS_DIR
    / "five_seed_validation_selected_fusion_oof_predictions.csv",
    index=False,
)

selected_pooled.to_csv(
    TABLES_DIR
    / "five_seed_validation_selected_pooled_metrics.csv",
    index=False,
)

selected_fold_metrics.to_csv(
    TABLES_DIR
    / "five_seed_validation_selected_fold_metrics.csv",
    index=False,
)


## 10. Paired bootstrap comparisons on the 544 OOF participants

Primary comparisons:

- equal SVM+TMC vs five-seed SVM ensemble;
- equal SVM+TMC vs five-seed TMC ensemble;
- validation-selected fusion vs five-seed SVM ensemble;
- validation-selected fusion vs equal fusion.


In [ ]:
def paired_bootstrap_auc_difference(
    y,
    p_a,
    p_b,
    n_bootstrap=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
):
    y = np.asarray(
        y,
        dtype=int,
    )

    p_a = np.asarray(
        p_a,
        dtype=float,
    )

    p_b = np.asarray(
        p_b,
        dtype=float,
    )

    idx0 = np.flatnonzero(
        y == 0
    )

    idx1 = np.flatnonzero(
        y == 1
    )

    rng = np.random.default_rng(
        seed
    )

    diffs = np.empty(
        n_bootstrap,
        dtype=float,
    )

    for b in range(
        n_bootstrap
    ):
        s0 = rng.choice(
            idx0,
            size=len(idx0),
            replace=True,
        )

        s1 = rng.choice(
            idx1,
            size=len(idx1),
            replace=True,
        )

        idx = np.concatenate(
            [
                s0,
                s1,
            ]
        )

        diffs[b] = (
            roc_auc_score(
                y[idx],
                p_a[idx],
            )
            - roc_auc_score(
                y[idx],
                p_b[idx],
            )
        )

    observed = (
        roc_auc_score(
            y,
            p_a,
        )
        - roc_auc_score(
            y,
            p_b,
        )
    )

    return {
        "AUC_DIFF":
            float(observed),

        "CI_LOW":
            float(
                np.quantile(
                    diffs,
                    0.025,
                )
            ),

        "CI_HIGH":
            float(
                np.quantile(
                    diffs,
                    0.975,
                )
            ),

        "BOOTSTRAP_SD":
            float(
                np.std(
                    diffs,
                    ddof=1,
                )
            ),

        "P_DIFF_GT_0":
            float(
                np.mean(
                    diffs > 0
                )
            ),
    }

comparisons = [
    (
        "equal_svm_tmc_5seed_ensemble",
        "svm_5seed_ensemble",
    ),
    (
        "equal_svm_tmc_5seed_ensemble",
        "tmc_5seed_ensemble",
    ),
    (
        "validation_selected_svm_tmc_5seed_ensemble",
        "svm_5seed_ensemble",
    ),
    (
        "validation_selected_svm_tmc_5seed_ensemble",
        "equal_svm_tmc_5seed_ensemble",
    ),
]

bootstrap_rows = []

for i, (
    method_a,
    method_b,
) in enumerate(
    comparisons
):
    result = paired_bootstrap_auc_difference(
        selected_oof["TARGET"],
        selected_oof[
            selected_method_cols[
                method_a
            ]
        ],
        selected_oof[
            selected_method_cols[
                method_b
            ]
        ],
        seed=(
            BOOTSTRAP_SEED
            + i
        ),
    )

    bootstrap_rows.append({
        "METHOD_A":
            method_a,

        "METHOD_B":
            method_b,

        **result,
    })

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_results
)

bootstrap_results.to_csv(
    TABLES_DIR
    / "paired_bootstrap_auc_differences.csv",
    index=False,
)


## 11. Visualize fold performance and selected SVM weights


In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 5)
)

for method in [
    "svm_5seed_ensemble",
    "tmc_5seed_ensemble",
    "equal_svm_tmc_5seed_ensemble",
    "validation_selected_svm_tmc_5seed_ensemble",
]:
    g = (
        selected_fold_metrics.loc[
            selected_fold_metrics["METHOD"]
            == method
        ]
        .sort_values(
            "FOLD"
        )
    )

    ax.plot(
        g["FOLD"],
        g["ROC_AUC"],
        marker="o",
        label=method,
    )

ax.set_xlabel(
    "Outer fold"
)

ax.set_ylabel(
    "ROC AUC"
)

ax.set_title(
    "Five-seed SVM–TMC performance by outer fold"
)

ax.set_xticks(
    FOLDS
)

ax.legend()
fig.tight_layout()

fold_fig = (
    FIGURES_DIR
    / "five_seed_svm_tmc_auc_by_fold.png"
)

fig.savefig(
    fold_fig,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

fig, ax = plt.subplots(
    figsize=(8, 4)
)

ax.plot(
    alpha_summary["FOLD"],
    alpha_summary[
        "SELECTED_ALPHA_SVM"
    ],
    marker="o",
)

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
)

ax.set_xlabel(
    "Outer fold"
)

ax.set_ylabel(
    "Validation-selected SVM weight"
)

ax.set_title(
    "Five-seed validation-selected fusion weight"
)

ax.set_xticks(
    FOLDS
)

ax.set_ylim(
    -0.02,
    1.02,
)

fig.tight_layout()

alpha_fig = (
    FIGURES_DIR
    / "five_seed_selected_alpha_by_fold.png"
)

fig.savefig(
    alpha_fig,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

print("Saved:", fold_fig)
print("Saved:", alpha_fig)


## 12. Final compact summary

The key outputs to report back are:

1. `matched_5seed_mean_sd.csv`  
   — mean ± SD across the five independent matched seeds.

2. `five_seed_ensemble_pooled_metrics.csv`  
   — the direct five-seed ensemble result across all 544 OOF participants.

3. `validation_selected_alpha_by_fold.csv`  
   — whether the preferred SVM weight is stable across folds.

4. `paired_bootstrap_auc_differences.csv`  
   — whether fusion improves over the five-seed SVM ensemble beyond sampling uncertainty.


In [ ]:
print("=" * 92)
print("FINAL FIVE-SEED, FIVE-FOLD SVM + TMC SUMMARY")
print("=" * 92)

print("\nA) Mean ± SD across five matched seeds:")
display(
    seed_summary
)

print("\nB) Five-seed probability-ensemble pooled OOF performance:")
display(
    selected_pooled
)

print("\nC) Validation-selected SVM alpha by fold:")
display(
    alpha_summary
)

print("\nD) Paired bootstrap AUC comparisons:")
display(
    bootstrap_results
)

best_row = (
    selected_pooled
    .sort_values(
        "ROC_AUC",
        ascending=False,
    )
    .iloc[0]
)

print(
    "\nBest five-seed ensemble method:",
    best_row["METHOD"],
)

print(
    "Best pooled OOF ROC AUC:",
    f"{best_row['ROC_AUC']:.6f}",
)

print(
    "\nAll pooled five-seed ensemble metrics are calculated "
    "over exactly 544 unique held-out participants."
)

print(
    "Seed sensitivity is reported separately as mean ± SD "
    "across seeds 17, 42, 73, 101 and 137."
)

print("\nOutputs saved to:", OUTPUT_ROOT)
